# Lecture d'un historique MARTHE

## lecture des données

In [ ]:
import pandas as pd

In [ ]:
df_obs = pd.read_csv(
    'data/00471X0095.csv',
    sep=',',
    encoding='utf-8',
    parse_dates=True,
    index_col=0
) #id(df) pour vérifier si même objet en memoire

In [ ]:
df_sim = pd.read_csv(
    'data/historiq.prn',
    encoding='ISO-8859-1',
    sep=r'\t',
    skiprows=[0],
    header=list(range(4)),
    parse_dates=True,
    index_col=0,
    dayfirst=True
)

df_sim.columns.names = ['variable', 'xy', 'grid', 'name']
df_sim.index.name = 'date'
df_sim

## extraction de données

In [ ]:
h_sim = df_sim.xs(level='variable', key=('Charge'), axis='columns')#.squeeze()
h_sim
#h_sim.name = 'Simulation'

In [ ]:
h_sim1 = df_sim.loc[:, df_sim.columns.get_level_values('variable').isin(['Charge'])].squeeze()#, , 'Débit_Rivi'
#h_sim
h_sim = df_sim.loc[:, ('Charge', slice(None), slice(None), '00471X0010/H1')] # slice(None, None, None) départ, fin, pas
h_sim.columns = ['Simulation']
h_sim

In [ ]:
h_obs = df_obs['value'][df_sim.index.min():df_sim.index.max()]
h_obs.name = 'Observation'
h_obs = h_obs.resample('D').mean()

## Plot

In [ ]:
from matplotlib import pyplot as plt
fig, ax = plt.subplots()
h_obs.plot(ax=ax, marker='.', markersize=2, color='k')
h_sim.plot(ax=ax, color='r')
ax.set_ylabel('hydraulic head [m]')
ax.grid()
ax.legend()

## calcul de scores

In [ ]:
df = pd.concat([h_obs, h_sim], axis=1)
print(df)

In [ ]:
df_skill = df.dropna(how='any', axis=0)

In [ ]:
df_skill

In [ ]:
df_skill.corr()

In [ ]:
means = df_skill.mean(axis=0)
(means['Simulation'] - means['Observation']).round(2)

In [ ]:
import numpy as np
rmse = np.sqrt(((df_skill['Simulation'] - df_skill['Observation'])**2).mean())
rmse

In [ ]:
import numpy as np
obs = df_skill['Observation']
sim = df_skill['Simulation']
nash = 1 - np.sum((sim - obs)**2) /np.sum((obs - obs.mean())**2)
nash.round(2)

In [ ]:
# q_sim = df_sim.xs(level='variable', key='Débit_Rivi', axis='columns').squeeze()
q_sim = df_sim.loc[:, ('Débit_Rivi', slice(None), slice(None), 'E6470092')]
q_sim.name = 'Simulation'
q_sim.plot(grid=True, title='Débit simulé', ylabel='m3/s')

In [ ]:
q_sim.groupby(h_sim.index.month).mean().plot(
    grid=True, title='Cycle saisonnier mensuel', style='-o', ylabel='m3/s'
)

## introduction à martpy

Martpy est un package python développé par le BRGM pour faciliter l'utilisation de Marthe dans un environnement python.
Cette librairie fait suite à de nombreux développements (librairies internes brgm, projet pymarthe, etc.) et
est actuellement en cours de développement (beta - 03/2026).

`martpy` permet de lire les fichiers d'entrées et de sorties de MARTHE.

Cette partie n'est pas accessible sur jupyterlite.

In [ ]:
# import martpy as mart

In [ ]:
# mart.read_prn('data/historiq.prn')

In [ ]:
# mart.read_prn('data/histobil_debit.prn')